# 06 — Statistical Rigor: Confidence Intervals + Walk-Forward CV

This notebook
answers the "how confident / how robust" question for the Exp 0 / A / B / C comparison built in
`03_model_training.ipynb` (Exp 0 ELO baseline, Exp A ELO-free XGBoost, Exp B ELO-included
XGBoost) and `05_gru_neural_elo.ipynb` (Exp C ELO-free GRU "neural ELO", hygiene-fixed
train-only `LIGAKOEF_SCALE`, calibrated test Brier ≈ 0.16278).

**Part 1 — Bootstrap confidence intervals.** 95% CIs (2000 resamples, seed 42) for accuracy,
Brier (primary), log-loss, AUC, for each of the four experiments, on the SAME held-out
time-based TEST set already used everywhere else in this project (3,426 matches / 6,852
duplicate-and-flip rows, 2025-01-19 .. 2026-07-08).

**CRITICAL — match-level resampling, not row-level.** The test set is duplicate-and-flip (2 rows
per match: `winner_pov` + `loser_pov`, same `SusretId`). Resampling individual ROWS with
replacement would let the two perspective-rows of the same match end up in different bootstrap
draws at different multiplicities, which is not how the real uncertainty in this dataset works —
a match's outcome is one random event, not two independent ones. Every resample here draws
**match IDs** with replacement and pulls **both** rows for each drawn match, preserving the paired
structure and giving an honest (not understated) variance estimate.

**Part 2 — Paired bootstrap on Brier differences.** Using the SAME match-level resamples (so the
comparisons are properly paired, not independently resampled), compute the distribution of
`Brier(exp_1) - Brier(exp_2)` for (C−0), (C−B), (B−0), (A−0). A 95% CI that excludes 0 is
evidence the improvement is statistically significant, not noise.

**Part 3 — Walk-forward (expanding-window) time-series CV.** Random k-fold is forbidden here
(temporal leakage — outcomes depend on past ELO/H2H/forma, all derived from historical order).
Instead: 5 equal chronological blocks of matches → 4 expanding-window folds (fold *k* trains on
blocks 1..*k*, tests on block *k*+1). Fold 4 is, by construction, numerically identical to the
project's main 80/20 split (train 13,702 / test 3,426 matches, cutoff 2025-01-19) — a built-in
sanity anchor. For each fold: Exp 0 is the closed-form ELO formula (no fitting); Exp A/B are
retrained XGBoost using the exact tuned hyperparameters and core/eval/calib chronological carve
from `03_model_training.ipynb`; Exp C is a retrained `TwoTowerGRU` using the exact architecture,
hyperparameters, and seed from `05_gru_neural_elo.ipynb`.

**Point-in-time note (important simplification, not a shortcut):** none of the engineered
features need to be recomputed per fold. Every feature in `features.parquet` (Exp A/B) and every
per-step GRU feature (Exp C) is already computed strictly as of its own row's match date — a
property established once in `02_feature_engineering.ipynb`'s leakage audit and re-verified in
`05_gru_neural_elo.ipynb` Step 5's spot-check. Reslicing which matches count as "fold train" vs
"fold test" by date does not change any single row's own feature values, so the SAME precomputed
frames are reused across all 4 folds — only the train/test **row masks** change per fold. This
is exactly the same reasoning already used once for the GRU's shared `HIST_ARRAY` (fold-independent
by construction) and is restated here because it looks like it could be a leakage risk and isn't.

**Simplification, documented:** the GRU's `LIGAKOEF_SCALE` input-scaling constant is held fixed at
the global max (`8.0`) across all 4 folds rather than recomputed per fold from each fold's own
train slice. `05_gru_neural_elo.ipynb` Step 15 already demonstrated (retrain-and-compare, not just
asserted) that this scale choice is numerically inert — a GRU's first-layer weights absorb any
fixed linear rescale of an input dimension — so repeating that recomputation 4 more times here
would add cost for no signal. One consequence, visible below: Fold 4's Exp C number in this
notebook (`LIGAKOEF_SCALE=8`, un-fixed) is ~0.00024 Brier away from `05`'s final train-only-scale
number (0.16278) — exactly the gap `05` itself already measured and documented as inert.

Artifacts: none saved outside this notebook (all inputs are the already-gitignored
`../data/test_predictions_4way.parquet`, `../data/features.parquet`, `../data/matches_export.csv`,
`../data/feature_tags.json` produced by notebooks 02/03/05).

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from torch.nn.utils.rnn import pack_padded_sequence

warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')

def compute_metrics_dict(y_true, p):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    out = {
        'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
        'brier': brier_score_loss(y_true, p),
        'log_loss': log_loss(y_true, p_c),
    }
    try:
        out['auc'] = roc_auc_score(y_true, p)
    except ValueError:
        out['auc'] = np.nan
    return out

print('Setup OK.')

## Part 1/2 — Load the 4-way TEST predictions (from notebooks 03 + 05)

`test_predictions_4way.parquet` already contains, for the exact same 6,852-row TEST set:
`p_exp0_elo`, `p_expA_calibrated`, `p_expB_calibrated`, `p_expC_calibrated` (plus the raw
uncalibrated variants). No retraining needed for Part 1/2 — this is purely a resampling exercise
on already-produced, already-calibrated predictions.

In [ ]:
preds = pd.read_parquet(DATA_DIR / 'test_predictions_4way.parquet').reset_index(drop=True)

EXPERIMENTS = {
    'Exp 0 - ELO baseline':          'p_exp0_elo',
    'Exp A - ELO-free XGBoost':      'p_expA_calibrated',
    'Exp B - ELO-included XGBoost':  'p_expB_calibrated',
    'Exp C - ELO-free GRU':          'p_expC_calibrated',
}

# Match-level grouping: every SusretId must have exactly 2 rows (winner_pov + loser_pov)
groups = preds.groupby('SusretId').indices
match_ids = np.array(list(groups.keys()))
lens = np.array([len(groups[m]) for m in match_ids])
assert (lens == 2).all(), 'expected exactly 2 rows (both perspectives) per match'
row_idx_2d = np.array([groups[m] for m in match_ids], dtype=np.int64)  # shape (n_matches, 2)
n_matches = len(match_ids)

y_all = preds['Label'].values
p_all = {name: preds[col].values for name, col in EXPERIMENTS.items()}

print(f"TEST set: {len(preds):,} rows, {n_matches:,} unique matches "
      f"({preds['Datum'].min().date()} .. {preds['Datum'].max().date()})")

point_estimates = {name: compute_metrics_dict(y_all, p_all[name]) for name in EXPERIMENTS}
pd.DataFrame(point_estimates).T.round(5)

### Match-level bootstrap: 2,000 resamples, both perspective rows travel together

For each of 2,000 iterations: draw `n_matches` match-IDs **with replacement**, pull BOTH rows
(`winner_pov` + `loser_pov`) for every drawn match ID, and recompute all four metrics for all four
experiments on that resampled set — plus, in the SAME iteration (so the differences are properly
paired), the Brier-score differences (C−0), (C−B), (B−0), (A−0).

In [ ]:
N_BOOT = 2000
rng = np.random.RandomState(RANDOM_STATE)

boot_metrics = {name: {'accuracy': [], 'brier': [], 'log_loss': [], 'auc': []} for name in EXPERIMENTS}
boot_diff = {'C_minus_0': [], 'C_minus_B': [], 'B_minus_0': [], 'A_minus_0': []}

t0 = time.time()
for b in range(N_BOOT):
    sampled_match_pos = rng.randint(0, n_matches, size=n_matches)
    idx = row_idx_2d[sampled_match_pos].reshape(-1)
    y_b = y_all[idx]

    briers = {}
    for name in EXPERIMENTS:
        p_b = p_all[name][idx]
        m = compute_metrics_dict(y_b, p_b)
        for k in boot_metrics[name]:
            boot_metrics[name][k].append(m[k])
        briers[name] = m['brier']

    boot_diff['C_minus_0'].append(briers['Exp C - ELO-free GRU'] - briers['Exp 0 - ELO baseline'])
    boot_diff['C_minus_B'].append(briers['Exp C - ELO-free GRU'] - briers['Exp B - ELO-included XGBoost'])
    boot_diff['B_minus_0'].append(briers['Exp B - ELO-included XGBoost'] - briers['Exp 0 - ELO baseline'])
    boot_diff['A_minus_0'].append(briers['Exp A - ELO-free XGBoost'] - briers['Exp 0 - ELO baseline'])

print(f'Match-level bootstrap done: {N_BOOT:,} resamples in {time.time()-t0:.1f}s (seed={RANDOM_STATE})')

## Part 1 — 95% Bootstrap CI table (all 4 experiments)

In [ ]:
rows = []
for name in EXPERIMENTS:
    row = {'experiment': name}
    for metric in ['accuracy', 'brier', 'log_loss', 'auc']:
        arr = np.array(boot_metrics[name][metric])
        lo, hi = np.percentile(arr, [2.5, 97.5])
        row[f'{metric}_point'] = point_estimates[name][metric]
        row[f'{metric}_ci_lo'] = lo
        row[f'{metric}_ci_hi'] = hi
    rows.append(row)

ci_table = pd.DataFrame(rows).set_index('experiment')
ci_table_display = ci_table.round(5)
ci_table_display

In [ ]:
def fmt_ci(point, lo, hi, dp=5):
    return f"{point:.{dp}f}  [{lo:.{dp}f}, {hi:.{dp}f}]"

print(f"{'experiment':32s} {'accuracy':28s} {'brier (primary)':28s} {'log_loss':28s} {'auc':28s}")
for name in EXPERIMENTS:
    r = ci_table.loc[name]
    print(f"{name:32s} "
          f"{fmt_ci(r['accuracy_point'], r['accuracy_ci_lo'], r['accuracy_ci_hi'], 4):28s} "
          f"{fmt_ci(r['brier_point'], r['brier_ci_lo'], r['brier_ci_hi']):28s} "
          f"{fmt_ci(r['log_loss_point'], r['log_loss_ci_lo'], r['log_loss_ci_hi']):28s} "
          f"{fmt_ci(r['auc_point'], r['auc_ci_lo'], r['auc_ci_hi'], 4):28s}")

## Part 2 — Paired bootstrap on Brier DIFFERENCES (significance)

Same 2,000 match-level resamples as Part 1 (not a separate draw), so each difference is a proper
paired comparison. A 95% CI that excludes 0 means the sign of the difference is consistent across
essentially all resamples — strong evidence the improvement (or regression) is real, not noise.

In [ ]:
diff_labels = {
    'C_minus_0': 'Exp C (GRU) minus Exp 0 (ELO)  -- does C beat the ELO baseline?',
    'C_minus_B': 'Exp C (GRU) minus Exp B (ELO-incl. XGB)  -- does C beat the strongest ELO-aware model?',
    'B_minus_0': 'Exp B (ELO-incl. XGB) minus Exp 0 (ELO)  -- does adding ELO to XGBoost beat plain ELO?',
    'A_minus_0': 'Exp A (ELO-free XGB) minus Exp 0 (ELO)  -- for reference/completeness',
}

diff_rows = []
for key, label in diff_labels.items():
    arr = np.array(boot_diff[key])
    lo, hi = np.percentile(arr, [2.5, 97.5])
    significant = not (lo < 0 < hi)
    direction = 'lower Brier (BETTER)' if arr.mean() < 0 else 'higher Brier (WORSE)'
    diff_rows.append({
        'comparison': key, 'description': label, 'mean_diff': arr.mean(),
        'ci_lo': lo, 'ci_hi': hi, 'significant_95pct': significant,
        'direction_of_mean': direction,
    })

diff_table = pd.DataFrame(diff_rows).set_index('comparison')
diff_table_display = diff_table.copy()
diff_table_display['mean_diff'] = diff_table_display['mean_diff'].round(5)
diff_table_display['ci_lo'] = diff_table_display['ci_lo'].round(5)
diff_table_display['ci_hi'] = diff_table_display['ci_hi'].round(5)
diff_table_display[['description', 'mean_diff', 'ci_lo', 'ci_hi', 'significant_95pct', 'direction_of_mean']]

In [ ]:
print('Verdicts (95% paired bootstrap CI on Brier differences, primary held-out TEST set):\n')
for key, label in diff_labels.items():
    r = diff_table.loc[key]
    verdict = 'YES -- statistically significant' if r['significant_95pct'] else 'NOT statistically significant (CI includes 0)'
    print(f"{label}")
    print(f"  mean diff = {r['mean_diff']:+.5f}   95% CI = [{r['ci_lo']:+.5f}, {r['ci_hi']:+.5f}]   -> {verdict}\n")

## Part 3 — Walk-forward (expanding-window) time-series CV

**Design:** all 17,128 in-scope matches, sorted chronologically, split into 5 equal blocks
(~3,425-3,426 matches each). 4 expanding-window folds: fold *k* trains on blocks `1..k`, tests on
block `k+1`. This means fold 4's train/test boundary (2025-01-19) is EXACTLY the project's
existing 80/20 split — a free, built-in sanity check that this replication pipeline reproduces
`03_model_training.ipynb` / `05_gru_neural_elo.ipynb` exactly before trusting folds 1-3.

A match is NEVER in both train and test within the same fold (blocks are disjoint by
construction), and both perspective rows (`winner_pov`/`loser_pov`) of a match always land in the
same bucket (train vs test) because the split is done at `SusretId` level, not row level — same
discipline as `03`/`05`.

In [ ]:
df = pd.read_parquet(DATA_DIR / 'features.parquet')
with open(DATA_DIR / 'feature_tags.json', encoding='utf-8') as f:
    TAGS = json.load(f)
EXP_A_COLS = TAGS['experiment_a_cols']
EXP_B_COLS = TAGS['experiment_b_cols']
df['liga_tip'] = df['liga_tip'].map({'SM': 0, 'SF': 1}).astype(int)
df = df.sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)

# Reuse the exact tuned hyperparameters from 03_model_training.ipynb (Step 4 CV) --
# no per-fold re-tuning, a pragmatic choice to isolate the walk-forward effect itself.
BEST_PARAMS_A = {'max_depth': 6, 'reg_alpha': 0.5, 'reg_lambda': 2.0, 'learning_rate': 0.05}
BEST_PARAMS_B = {'max_depth': 4, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'learning_rate': 0.05}

match_order_all = (df[['SusretId', 'Datum']].drop_duplicates()
                   .sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True))
n_matches_all = len(match_order_all)
N_BLOCKS = 5
block_edges = np.linspace(0, n_matches_all, N_BLOCKS + 1).astype(int)
blocks = [match_order_all.iloc[block_edges[i]:block_edges[i + 1]] for i in range(N_BLOCKS)]

FOLDS = []
for k in range(1, N_BLOCKS):
    train_ids = set(pd.concat(blocks[:k])['SusretId'])
    test_ids = set(blocks[k]['SusretId'])
    cutoff_date = blocks[k]['Datum'].min()
    FOLDS.append({'fold': k, 'train_ids': train_ids, 'test_ids': test_ids, 'cutoff': cutoff_date})
    print(f"Fold {k}: train={len(train_ids):,} matches, test={len(test_ids):,} matches, "
          f"cutoff={cutoff_date.date()}")

assert FOLDS[-1]['train_ids'].union(FOLDS[-1]['test_ids']).issuperset(set(match_order_all['SusretId']))
print(f"\nFold 4 train count ({len(FOLDS[-1]['train_ids']):,}) should equal "
      f"the project's main split train count ({TAGS['n_matches_train']:,}) -- sanity anchor.")
assert len(FOLDS[-1]['train_ids']) == TAGS['n_matches_train']
assert len(FOLDS[-1]['test_ids']) == TAGS['n_matches_test']
print('OK -- fold 4 exactly reproduces the main 80/20 split boundary.')

### Shared GRU history precompute (built ONCE, reused by every fold)

Identical construction to `05_gru_neural_elo.ipynb` Steps 1-5 (long-format player log,
per-step point-in-time features, `MAX_LEN=24` capped history with the debut placeholder step).
As explained in the intro above, this is fold-independent by construction — a row's own history
is always strictly-prior matches, regardless of which fold later treats that row as train or
test.

In [ ]:
raw = pd.read_csv(DATA_DIR / 'matches_export.csv', parse_dates=['Datum'])
raw['Datum'] = pd.to_datetime(raw['Datum']).dt.tz_localize(None)
raw = raw.sort_values(['Datum', 'SusretId']).reset_index(drop=True)

common_cols = ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'LigaTip', 'LigaKoeficjent', 'Lokacija']
winner_rows = raw[common_cols + ['WinnerId', 'LoserId', 'W_Set1', 'W_Set2', 'W_Set3', 'L_Set1', 'L_Set2', 'L_Set3']].rename(
    columns={'WinnerId': 'IgracId', 'LoserId': 'OpponentId',
             'W_Set1': 'MySet1', 'W_Set2': 'MySet2', 'W_Set3': 'MySet3',
             'L_Set1': 'OppSet1', 'L_Set2': 'OppSet2', 'L_Set3': 'OppSet3'}
).assign(Won=1)
loser_rows = raw[common_cols + ['LoserId', 'WinnerId', 'L_Set1', 'L_Set2', 'L_Set3', 'W_Set1', 'W_Set2', 'W_Set3']].rename(
    columns={'LoserId': 'IgracId', 'WinnerId': 'OpponentId',
             'L_Set1': 'MySet1', 'L_Set2': 'MySet2', 'L_Set3': 'MySet3',
             'W_Set1': 'OppSet1', 'W_Set2': 'OppSet2', 'W_Set3': 'OppSet3'}
).assign(Won=0)
player_log = pd.concat([winner_rows, loser_rows], ignore_index=True)
player_log['LokacijaNorm'] = player_log['Lokacija'].str.strip().str.lower()
player_log = player_log.sort_values(['IgracId', 'Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)

g = player_log.groupby('IgracId', sort=False)
player_log['MatchCountBefore'] = g.cumcount()
player_log['HomeCountBefore'] = player_log.groupby(['IgracId', 'LokacijaNorm'], sort=False).cumcount()
player_log['HomeShareBefore'] = np.where(
    player_log['MatchCountBefore'] > 0,
    player_log['HomeCountBefore'] / player_log['MatchCountBefore'], np.nan)
player_log['DaysSinceLastMatch'] = player_log.groupby('IgracId')['Datum'].diff().dt.days
player_log['CareerWinRateBefore'] = g['Won'].transform(lambda s: s.shift().expanding().mean())

for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']:
    player_log[c + '_f'] = player_log[c].fillna(0)
player_log['SetMargin'] = (
    (player_log['MySet1_f'] + player_log['MySet2_f'] + player_log['MySet3_f'])
    - (player_log['OppSet1_f'] + player_log['OppSet2_f'] + player_log['OppSet3_f']))
player_log = player_log.drop(columns=[c + '_f' for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']])

opp_cwr = player_log[['SusretId', 'IgracId', 'CareerWinRateBefore']].rename(
    columns={'IgracId': 'OpponentId', 'CareerWinRateBefore': 'OpponentCareerWinRateBefore'})
player_log = player_log.merge(opp_cwr, on=['SusretId', 'OpponentId'], how='left', validate='many_to_one')

MAX_LEN = 24
N_FEATS = 7
SETMARGIN_SCALE = 12.0
DAYSREST_CLIP = 90.0
# Fixed at the global max across ALL folds -- documented as inert in 05 Step 15 (see intro above).
LIGAKOEF_SCALE_GLOBAL = float(player_log['LigaKoeficjent'].max())

step_feats = np.zeros((len(player_log), N_FEATS), dtype=np.float32)
step_feats[:, 0] = player_log['Won'].values
step_feats[:, 1] = np.clip(player_log['SetMargin'].values / SETMARGIN_SCALE, -3, 3)
days_rest = player_log['DaysSinceLastMatch'].fillna(DAYSREST_CLIP).clip(upper=DAYSREST_CLIP).values
step_feats[:, 2] = days_rest / DAYSREST_CLIP
step_feats[:, 3] = player_log['LigaKoeficjent'].values / LIGAKOEF_SCALE_GLOBAL
step_feats[:, 4] = player_log['HomeShareBefore'].fillna(0.0).values
step_feats[:, 5] = player_log['OpponentCareerWinRateBefore'].fillna(0.5).values
step_feats[:, 6] = 0.0

PLACEHOLDER_VEC = np.zeros(N_FEATS, dtype=np.float32)
PLACEHOLDER_VEC[5] = 0.5
PLACEHOLDER_VEC[6] = 1.0

player_log['_gpos'] = np.arange(len(player_log))
first_idx = player_log.groupby('IgracId')['_gpos'].min().to_dict()
idx_lookup = dict(zip(zip(player_log['IgracId'], player_log['SusretId']), player_log['_gpos']))

def build_history(igrac_id, susret_id):
    gpos = idx_lookup[(igrac_id, susret_id)]
    start = first_idx[igrac_id]
    p = gpos - start
    seq = np.zeros((MAX_LEN, N_FEATS), dtype=np.float32)
    if p == 0:
        seq[0] = PLACEHOLDER_VEC
        length = 1
    else:
        lo = max(start, gpos - MAX_LEN)
        chunk = step_feats[lo:gpos]
        length = chunk.shape[0]
        seq[:length] = chunk
    return seq, length

n_rows_pl = len(player_log)
HIST_ARRAY = np.zeros((n_rows_pl, MAX_LEN, N_FEATS), dtype=np.float32)
LEN_ARRAY = np.zeros(n_rows_pl, dtype=np.int64)
igrac_arr, susret_arr = player_log['IgracId'].values, player_log['SusretId'].values
for i in range(n_rows_pl):
    HIST_ARRAY[i], LEN_ARRAY[i] = build_history(igrac_arr[i], susret_arr[i])

HIST_T = torch.from_numpy(HIST_ARRAY)
LEN_T = torch.from_numpy(LEN_ARRAY)

sym = df[['SusretId', 'Datum', 'PlayerId', 'OpponentId', 'Label', 'Perspective']].copy()
sym = sym.sort_values(['Datum', 'SusretId', 'Perspective'], kind='mergesort').reset_index(drop=True)
player_pos = np.array([idx_lookup[(pid, sid)] for pid, sid in zip(sym['PlayerId'], sym['SusretId'])])
opp_pos = np.array([idx_lookup[(oid, sid)] for oid, sid in zip(sym['OpponentId'], sym['SusretId'])])

print(f'Precomputed histories for {n_rows_pl:,} player-match rows (shared across all 4 folds).')
print(f'LIGAKOEF_SCALE_GLOBAL = {LIGAKOEF_SCALE_GLOBAL}')

### Per-fold training helpers (chronological carve, XGBoost, GRU)

Same `calib_frac=0.15` / `early_stop_frac=0.10` chronological carve within each fold's OWN train
slice as `03_model_training.ipynb` / `05_gru_neural_elo.ipynb`, same hyperparameters, same
`TwoTowerGRU` architecture, same seed (42).

In [ ]:
def carve_train(train_match_ids, all_match_order, calib_frac=0.15, early_stop_frac=0.10):
    fold_order = all_match_order[all_match_order['SusretId'].isin(train_match_ids)].reset_index(drop=True)
    n = len(fold_order)
    n_calib = max(1, int(round(n * calib_frac)))
    fit_order = fold_order.iloc[: n - n_calib].reset_index(drop=True)
    calib_ids = set(fold_order.iloc[n - n_calib:]['SusretId'])
    n_fit = len(fit_order)
    n_eval = max(1, int(round(n_fit * early_stop_frac)))
    core_ids = set(fit_order.iloc[: n_fit - n_eval]['SusretId'])
    eval_ids = set(fit_order.iloc[n_fit - n_eval:]['SusretId'])
    return core_ids, eval_ids, calib_ids


def fit_xgb_fold(cols, best_params, core_mask, eval_mask, calib_mask, test_mask, n_estimators=500,
                  isotonic_min_rows=1000):
    X_core, y_core = df.loc[core_mask, cols], df.loc[core_mask, 'Label']
    X_eval, y_eval = df.loc[eval_mask, cols], df.loc[eval_mask, 'Label']
    X_calib, y_calib = df.loc[calib_mask, cols], df.loc[calib_mask, 'Label']
    X_test, y_test = df.loc[test_mask, cols], df.loc[test_mask, 'Label']

    model = xgb.XGBClassifier(
        n_estimators=n_estimators, max_depth=best_params['max_depth'],
        learning_rate=best_params['learning_rate'], reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'], subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_STATE, tree_method='hist', eval_metric='logloss',
        early_stopping_rounds=30, n_jobs=-1,
    )
    model.fit(X_core, y_core, eval_set=[(X_eval, y_eval)], verbose=False)

    calib_method = 'isotonic' if len(X_calib) >= isotonic_min_rows else 'sigmoid'
    calibrated = CalibratedClassifierCV(estimator=model, method=calib_method, cv='prefit')
    calibrated.fit(X_calib, y_calib)

    p_test = calibrated.predict_proba(X_test)[:, 1]
    return p_test, y_test.values


class TwoTowerGRU(nn.Module):
    def __init__(self, input_size=N_FEATS, hidden_size=24, dropout=0.3, mlp_hidden=16):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop_embed = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 3, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))

    def encode(self, seq, lengths):
        packed = pack_padded_sequence(seq, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.drop_embed(h_n[-1])

    def forward(self, player_seq, player_len, opp_seq, opp_len):
        h_p = self.encode(player_seq, player_len)
        h_o = self.encode(opp_seq, opp_len)
        combined = torch.cat([h_p, h_o, h_p - h_o], dim=1)
        return self.head(combined).squeeze(-1)


HIDDEN_SIZE, DROPOUT, WEIGHT_DECAY, LR = 24, 0.3, 1e-4, 2e-3
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 512, 60, 8


def bucket_tensors(mask_ids):
    mask = sym['SusretId'].isin(mask_ids).values
    pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
    op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
    y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
    return HIST_T[pp], LEN_T[pp], HIST_T[op], LEN_T[op], y


def fit_gru_fold(core_ids, eval_ids, calib_ids, test_ids, seed=RANDOM_STATE):
    core = bucket_tensors(core_ids)
    ev = bucket_tensors(eval_ids)
    calib = bucket_tensors(calib_ids)
    test = bucket_tensors(test_ids)

    torch.manual_seed(seed)
    model = TwoTowerGRU(hidden_size=HIDDEN_SIZE, dropout=DROPOUT)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    n_core = core[4].shape[0]
    rng_local = np.random.RandomState(seed)

    best_brier, best_state, no_improve = np.inf, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = rng_local.permutation(n_core)
        for start in range(0, n_core, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            logit = model(core[0][idx], core[1][idx], core[2][idx], core[3][idx])
            loss = criterion(logit, core[4][idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logit = model(ev[0], ev[1], ev[2], ev[3])
        val_brier = brier_score_loss(ev[4].numpy(), torch.sigmoid(val_logit).numpy())
        if val_brier < best_brier - 1e-5:
            best_brier, best_state, no_improve = val_brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break
    model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        calib_logit = model(calib[0], calib[1], calib[2], calib[3])
        test_logit = model(test[0], test[1], test[2], test[3])
    calib_logit_np, calib_y_np = calib_logit.numpy(), calib[4].numpy()
    test_logit_np, test_y_np = test_logit.numpy(), test[4].numpy()

    T_param = torch.nn.Parameter(torch.ones(1))
    clt, cyt = torch.from_numpy(calib_logit_np), torch.from_numpy(calib_y_np)
    opt_T = torch.optim.LBFGS([T_param], lr=0.05, max_iter=100)

    def _closure():
        opt_T.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(clt / T_param, cyt)
        loss.backward()
        return loss
    opt_T.step(_closure)
    T_value = float(T_param.detach().clamp(min=1e-2))

    p_calib_raw = 1 / (1 + np.exp(-calib_logit_np))
    p_calib_temp = torch.sigmoid(clt / T_value).numpy()
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(p_calib_raw, calib_y_np)
    p_calib_iso = iso.predict(p_calib_raw)

    p_test_raw = 1 / (1 + np.exp(-test_logit_np))
    if brier_score_loss(calib_y_np, p_calib_iso) < brier_score_loss(calib_y_np, p_calib_temp):
        p_test_cal = iso.predict(p_test_raw)
    else:
        p_test_cal = torch.sigmoid(torch.from_numpy(test_logit_np) / T_value).numpy()

    return p_test_cal, test_y_np

print('Fold-training helpers defined.')

### Run all 4 folds (Exp 0 / A / B / C each)

Budget: XGBoost fits in seconds per fold; the GRU is the heavy part (~30-160s per fold,
proportional to that fold's train-set size — fold 4 has the most data and takes the longest,
matching the ~136-156s already seen for the full dataset in `05_gru_neural_elo.ipynb`). Total
expected wall time for this cell: roughly 5-7 minutes on CPU.

In [ ]:
results = []
t_all = time.time()

for fspec in FOLDS:
    k, train_ids, test_ids, cutoff = fspec['fold'], fspec['train_ids'], fspec['test_ids'], fspec['cutoff']
    t0 = time.time()
    print(f"\n=== FOLD {k} (cutoff {cutoff.date()}, train={len(train_ids):,} matches, "
          f"test={len(test_ids):,} matches) ===")

    test_mask = df['SusretId'].isin(test_ids)
    y_test = df.loc[test_mask, 'Label'].values

    p0 = (1 / (1 + 10 ** (-df.loc[test_mask, 'elo_diff'] / 400))).values
    m0 = compute_metrics_dict(y_test, p0)
    print(f"  Exp0 (ELO, no fit)      brier={m0['brier']:.5f}  acc={m0['accuracy']:.4f}")

    core_ids, eval_ids, calib_ids = carve_train(train_ids, match_order_all)
    core_mask = df['SusretId'].isin(core_ids)
    eval_mask = df['SusretId'].isin(eval_ids)
    calib_mask = df['SusretId'].isin(calib_ids)

    pA, yA = fit_xgb_fold(EXP_A_COLS, BEST_PARAMS_A, core_mask, eval_mask, calib_mask, test_mask)
    mA = compute_metrics_dict(yA, pA)
    print(f"  ExpA (ELO-free XGB)     brier={mA['brier']:.5f}  acc={mA['accuracy']:.4f}  "
          f"({time.time()-t0:.1f}s elapsed)")

    pB, yB = fit_xgb_fold(EXP_B_COLS, BEST_PARAMS_B, core_mask, eval_mask, calib_mask, test_mask)
    mB = compute_metrics_dict(yB, pB)
    print(f"  ExpB (ELO-incl. XGB)    brier={mB['brier']:.5f}  acc={mB['accuracy']:.4f}  "
          f"({time.time()-t0:.1f}s elapsed)")

    pC, yC = fit_gru_fold(core_ids, eval_ids, calib_ids, test_ids)
    mC = compute_metrics_dict(yC, pC)
    print(f"  ExpC (ELO-free GRU)     brier={mC['brier']:.5f}  acc={mC['accuracy']:.4f}  "
          f"({time.time()-t0:.1f}s elapsed)")

    for m, expname in [(m0, 'Exp 0 - ELO baseline'), (mA, 'Exp A - ELO-free XGBoost'),
                        (mB, 'Exp B - ELO-included XGBoost'), (mC, 'Exp C - ELO-free GRU')]:
        results.append({'fold': k, 'experiment': expname, 'n_train_matches': len(train_ids),
                         'n_test_matches': len(test_ids), 'cutoff': str(cutoff.date()), **m})

    print(f"  Fold {k} total: {time.time()-t0:.1f}s")

walkforward_df = pd.DataFrame(results)
print(f"\nAll folds done in {time.time()-t_all:.1f}s")

## Part 3 — Walk-forward results: per-fold table + mean±std summary

In [ ]:
per_fold_pivot = walkforward_df.pivot(index='fold', columns='experiment', values='brier')
per_fold_pivot = per_fold_pivot[['Exp 0 - ELO baseline', 'Exp A - ELO-free XGBoost',
                                  'Exp B - ELO-included XGBoost', 'Exp C - ELO-free GRU']]
print('Per-fold Brier (lower = better):')
per_fold_pivot.round(5)

In [ ]:
summary = walkforward_df.groupby('experiment')[['accuracy', 'brier', 'log_loss', 'auc']].agg(['mean', 'std'])
summary = summary.reindex(['Exp 0 - ELO baseline', 'Exp A - ELO-free XGBoost',
                            'Exp B - ELO-included XGBoost', 'Exp C - ELO-free GRU'])
summary_display = summary.round(5)
print('Mean +/- std across the 4 walk-forward folds:')
summary_display

In [ ]:
print('Per-fold Brier ranking (best -> worst):')
ranking_stable = True
for f in sorted(walkforward_df['fold'].unique()):
    sub = walkforward_df[walkforward_df['fold'] == f].sort_values('brier')
    ranking = ' < '.join(sub['experiment'].str.replace(r' - .*', '', regex=True))
    print(f"  Fold {f}: {ranking}   (briers: {list(sub['brier'].round(5))})")

c_beats_0_per_fold = walkforward_df.pivot(index='fold', columns='experiment', values='brier').apply(
    lambda r: r['Exp C - ELO-free GRU'] < r['Exp 0 - ELO baseline'], axis=1)
c_beats_b_per_fold = walkforward_df.pivot(index='fold', columns='experiment', values='brier').apply(
    lambda r: r['Exp C - ELO-free GRU'] < r['Exp B - ELO-included XGBoost'], axis=1)
b_beats_0_per_fold = walkforward_df.pivot(index='fold', columns='experiment', values='brier').apply(
    lambda r: r['Exp B - ELO-included XGBoost'] < r['Exp 0 - ELO baseline'], axis=1)
a_beats_0_per_fold = walkforward_df.pivot(index='fold', columns='experiment', values='brier').apply(
    lambda r: r['Exp A - ELO-free XGBoost'] < r['Exp 0 - ELO baseline'], axis=1)

print(f"\nExp C beats Exp 0 (ELO) in {c_beats_0_per_fold.sum()}/4 folds: {dict(c_beats_0_per_fold)}")
print(f"Exp C beats Exp B in {c_beats_b_per_fold.sum()}/4 folds: {dict(c_beats_b_per_fold)}")
print(f"Exp B beats Exp 0 (ELO) in {b_beats_0_per_fold.sum()}/4 folds: {dict(b_beats_0_per_fold)}")
print(f"Exp A beats Exp 0 (ELO) in {a_beats_0_per_fold.sum()}/4 folds: {dict(a_beats_0_per_fold)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {'Exp 0 - ELO baseline': '#374151', 'Exp A - ELO-free XGBoost': '#dc2626',
          'Exp B - ELO-included XGBoost': '#2563eb', 'Exp C - ELO-free GRU': '#7c3aed'}
markers = {'Exp 0 - ELO baseline': 's', 'Exp A - ELO-free XGBoost': 'x',
           'Exp B - ELO-included XGBoost': '^', 'Exp C - ELO-free GRU': 'D'}
for exp in per_fold_pivot.columns:
    ax.plot(per_fold_pivot.index, per_fold_pivot[exp], label=exp, color=colors[exp],
            marker=markers[exp], linewidth=2)
ax.set_xlabel('Walk-forward fold (expanding window, later fold = more train data)')
ax.set_ylabel('Test Brier score (lower = better)')
ax.set_title('Walk-forward CV: Brier score per fold, all 4 experiments')
ax.set_xticks(per_fold_pivot.index)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../models/walkforward_brier_by_fold.png', dpi=120)
plt.show()

## Part 4 — Written Summary: Is Exp C's edge real, significant, and robust?

**Part 1 (bootstrap CIs) — headline numbers hold up, with real uncertainty width.** All four
experiments' point estimates from `03`/`05` sit comfortably inside their own 95% CIs (no
surprises), but the CI widths are non-trivial: Brier CIs span roughly ±0.006-0.007 for Exp
0/A/B/C individually. This confirms the original single point estimates were not made up, but
also that small differences between experiments need the PAIRED (not just individual) CIs to
interpret correctly — which is exactly what Part 2 does.

**Part 2 (paired bootstrap on differences) — the key sentence.** On the held-out TEST set
(N_BOOT=2,000, seed=42; exact numbers from this run's `diff_table` above):
- **Exp C (GRU) vs Exp 0 (ELO): mean Brier diff ≈ −0.0144, 95% CI ≈ [−0.0192, −0.0096] —
  entirely negative, excludes 0.** **Exp C beating ELO is statistically significant**, not noise.
- **Exp C (GRU) vs Exp B (ELO-included XGBoost): mean Brier diff ≈ −0.0042, 95% CI ≈
  [−0.0093, +0.0008] — straddles 0.** Despite Exp C's lower point-estimate Brier (0.16278 vs
  0.16699), the paired bootstrap says this specific gap is **NOT statistically distinguishable
  from zero** on this test set. This is the single most important finding of Part 2 for
  interpreting `05`'s written verdict: "Exp C is the strongest model" is true on point estimate,
  but "Exp C is *significantly* stronger than Exp B" is **not** supported at 95% confidence here.
- **Exp B vs Exp 0: mean Brier diff ≈ −0.0102, 95% CI ≈ [−0.0141, −0.0063] — significant.**
  Adding ELO to XGBoost reliably beats the plain ELO formula.
- **Exp A vs Exp 0: mean Brier diff ≈ +0.0197, 95% CI ≈ [+0.0131, +0.0264] — significant in the
  WORSE direction.** ELO-free XGBoost (static summary features) reliably underperforms plain
  ELO. Consistent with `04_model_evaluation.ipynb`.
(These are this scratchpad-validated run's numbers; the notebook re-derives them independently
with the same seed — the printed verdict block above is the authoritative source if the two
ever disagree by more than bootstrap noise.)

**Part 3 (walk-forward CV) — this is where the picture gets meaningfully more cautious.**
Read directly off `per_fold_pivot`/the ranking printout above:
- **Exp A > Exp 0 (worse) holds in all 4 folds** — the single most temporally stable finding in
  this entire notebook. Static-summary ELO-free XGBoost never beats ELO, in any era of the data.
- **Exp B < Exp 0 (better) holds in all 4 folds** — also fully stable. ELO-included XGBoost
  reliably improves on the raw ELO formula across every time period tested.
- **Exp C < Exp 0 (better) holds in only 3 of the 4 folds** — fold 1 (the earliest period, and
  the fold with the LEAST training data, ~3,425 matches) is the one exception: there, plain ELO
  actually beats the GRU. Read together with the small-data GRU literature, this points at a
  cold-start effect — the GRU needs a longer accumulated razina/results history before its
  edge over ELO materializes; with the least data of any fold, that edge has not yet appeared.
- **Exp C < Exp B (GRU beats the strongest XGBoost model) holds in only 1 of the 4 folds — fold 4,
  the exact same fold that is the project's main 80/20 split.** In folds 1, 2, and 3, Exp B
  (ELO-included XGBoost) is BETTER than Exp C, not worse. Averaged across all 4 folds (mean
  Brier), **Exp B (~0.1667) is essentially tied with or marginally ahead of Exp C (~0.1685)** —
  the opposite ranking from the single 80/20 split's headline result.

**Honest bottom line.** Exp C's win over the plain ELO baseline (Exp 0) is real and reasonably
robust: it is statistically significant on the primary test set (Part 2) and holds directionally
in 3 of 4 independent time folds (Part 3), failing only in the smallest/earliest-data fold — a
plausible cold-start effect, not random noise. **But the more exciting, headline claim from
`05_gru_neural_elo.ipynb` — that Exp C is the single best model, beating even the ELO-included
XGBoost (Exp B) — does NOT hold up under either form of validation here.** The paired bootstrap
CI on (C − B) includes 0, and the walk-forward ranking of C vs. B flips in 3 of 4 folds, with B
ahead on average. The razina-trajectory mechanism identified in `05`'s ablation (Step 15) is a
real, legitimate, ELO-free signal — the walk-forward result does not contradict that mechanism
existing — but this notebook's evidence is that **whether the GRU's exploitation of that signal
is good enough to beat a well-tuned ELO-aware XGBoost model appears to depend on how much
training history is available**, and is not yet a settled result. If a production decision were
being made today, Exp B (ELO-included XGBoost) and Exp C (ELO-free GRU) should be treated as
**statistically tied contenders for the top spot**, both clearly and robustly ahead of the plain
ELO formula and of ELO-free XGBoost — not "Exp C is the winner." This is a more conservative,
better-supported conclusion than the single 80/20 split alone could offer, and it is exactly the
kind of result the walk-forward check was built to catch.

## Part 5 — Update: Three Follow-Up Notebooks Extend This Picture (07, 08, 09)

*(Added after `07_xgboost_trajectory.ipynb`, `08_gru_elo.ipynb`, and `09_walkforward_top_models.ipynb` were built — the analysis above is unchanged and remains the correct read of the ORIGINAL four experiments, Exp 0/A/B/C.)*

This notebook's central finding — Exp C's single-split "beats everything" claim does not survive paired bootstrap (C vs B is not significant) or walk-forward CV (C loses to B in 3/4 folds) — turned out to be the right instinct to have, and it directly motivated the next three notebooks:

- **`07_xgboost_trajectory.ipynb`** asks WHY Exp C beats Exp A so much in the first place, and hands XGBoost the same signal (razina as a trajectory of 9 point-in-time features instead of a static snapshot). Result: **Exp A' (ELO-free XGB + trajectory) beats Exp 0 for the first time** (Brier 0.1754 vs 0.1771) — closing roughly two-thirds of the original ELO-free gap to Exp C, with a smaller, still-significant remainder left over for the GRU. Adding the same trajectory features on top of Exp B (Exp B') gives no significant further lift — consistent with ELO already encoding most of that signal.
- **`08_gru_elo.ipynb`** builds the natural counterpart this notebook's Part 4 summary was missing: Exp C2 (the ELO-free GRU from `05`, with point-in-time ELO added at the head, the direct GRU analog of Exp B). Exp C2 is the best point estimate of every experiment run in this project (Brier 0.1609) and — unlike Exp C here — its edge over Exp B AND Exp B' IS statistically significant. Its edge over the ELO-free Exp C, however, is NOT significant: ELO and the GRU's own learned representation are close substitutes for this dataset, not complements.
- **`09_walkforward_top_models.ipynb`** (capstone) re-runs the walk-forward design established here — same 5-block / 4-fold expanding-window structure — but on the two strongest successors (Exp B', Exp C2) instead of the original four. Exp C2 beats plain ELO in 4/4 folds (a fully robust result, unlike Exp C's 3/4 here — adding ELO at the head is exactly what rescues the cold-start fold this notebook flagged), and beats Exp B' in 3/4 folds, losing only in the earliest/thinnest-data fold — where Exp B' remains the single best model. That fold-1 exception mirrors the sample-efficiency caution this notebook already raised for Exp C's cold-start miss.

**Net effect on this notebook's own conclusion:** unchanged for the ORIGINAL four experiments — "Exp C vs Exp B is a statistical tie, both clearly ahead of ELO and ELO-free XGBoost" remains the correct read of that specific comparison. But it is no longer the final word on "what is the best model in this project": that question is answered, with the strongest available evidence, by `09`. See `../README.md` for the full seven-experiment table and narrative arc.
